# Chapitre 9 · L'attention (solutions des exercices)

Ce notebook contient **uniquement les réponses aux sept exercices** du notebook
du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut ici plus que partout ailleurs dans le livre. L'attention
se gagne à la main.

## Mise en place (reprise de la leçon)

Le minimum pour que les validations tournent : les imports et les données jouets.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

# Les données jouets de la leçon : 4 tokens, d_k = 4.
Q = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0.5, 0.5, 0, 0], [0, 0, 0.5, 0.5]], dtype=np.float32)
K = np.eye(4, dtype=np.float32)
V = np.array([[10, 0, 0, 0], [0, 20, 0, 0], [0, 0, 30, 0], [0, 0, 0, 40]], dtype=np.float32)

### Exercice 1 · Les scores mis à l'échelle — niveau ●

Étapes 1 et 2 de la formule : le produit scalaire de toutes les paires query/key, divisé par
$\sqrt{d_k}$. Rappel du réflexe `shape` : `(n, d_k) @ (d_k, n) = (n, n)`.

In [ ]:
def scores_attention(Q, K):
    """Étapes 1 + 2 : scores de toutes les paires, mis à l'échelle.

    Q : (n, d_k), K : (n, d_k)  ->  scores : (n, n)
    L'élément (i, j) se lit : « à quel point le token i s'intéresse au token j ».
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    return scores

In [ ]:
S = scores_attention(Q, K)
S_attendu = np.array([[0.500000, 0.000000, 0.000000, 0.000000],
              [0.000000, 0.500000, 0.000000, 0.000000],
              [0.250000, 0.250000, 0.000000, 0.000000],
              [0.000000, 0.000000, 0.250000, 0.250000]], dtype=np.float32)
assert S.shape == (4, 4), f"shape {S.shape} au lieu de (4, 4) : as-tu bien transposé K ?"
assert np.allclose(S, S_attendu, atol=1e-4), "valeurs inattendues : as-tu divisé par np.sqrt(d_k) et pas d_k ?"
print("Exercice 1 validé : les scores sont corrects.")
print(S)

### Exercice 2 · Le softmax stable, ligne par ligne — niveau ●●

Étape 3 : chaque ligne de scores devient une distribution de probabilité. Et souviens-toi de la
section 3.4 du chapitre : on soustrait le maximum de chaque ligne avant l'exponentielle
(le log-sum-exp trick), sinon `exp` déborde sur de grands scores.

In [ ]:
def softmax_stable(scores):
    """Étape 3 : softmax ligne par ligne, numériquement stable.

    scores : (n, n)  ->  weights : (n, n), chaque ligne positive et de somme 1.
    """
    scores = scores - scores.max(axis=1, keepdims=True)   # log-sum-exp : pas d'overflow
    exp_scores = np.exp(scores)
    weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    return weights

In [ ]:
W = softmax_stable(S)
W_attendu = np.array([[0.354661, 0.215113, 0.215113, 0.215113],
              [0.215113, 0.354661, 0.215113, 0.215113],
              [0.281088, 0.281088, 0.218912, 0.218912],
              [0.218912, 0.218912, 0.281088, 0.281088]], dtype=np.float32)
assert np.allclose(W.sum(axis=1), 1.0, atol=1e-5), "chaque ligne doit sommer à 1"
assert np.allclose(W, W_attendu, atol=1e-4), "valeurs inattendues : softmax appliqué ligne par ligne (axis=1) ?"
assert W[0].argmax() == 0, "le token 0 devrait regarder surtout la key 0 (son meilleur match)"
grands = softmax_stable(np.array([[1000.0, 1001.0, 999.0]]))
assert np.isfinite(grands).all(), "overflow : as-tu soustrait le max de chaque ligne avant np.exp ?"
print("Exercice 2 validé : softmax correct et stable, même sur des scores énormes.")
print(np.round(W, 3))

### Exercice 3 · La moyenne pondérée des values — niveau ●

Étape 4 : chaque token compose son nouveau vecteur en mélangeant les values de tous les tokens,
chacune pondérée par l'attention qu'il lui porte.

In [ ]:
def melange_values(weights, V):
    """Étape 4 : output = weights @ V.

    weights : (n, n), V : (n, d_v)  ->  output : (n, d_v)
    """
    output = weights @ V
    return output

In [ ]:
O = melange_values(W, V)
O_attendu = np.array([[3.5466, 4.3023, 6.4534, 8.6045],
              [2.1511, 7.0932, 6.4534, 8.6045],
              [2.8109, 5.6218, 6.5674, 8.7565],
              [2.1891, 4.3782, 8.4326, 11.2435]], dtype=np.float32)
assert O.shape == (4, 4), f"shape {O.shape} au lieu de (4, 4)"
assert np.allclose(O, O_attendu, atol=1e-2), "valeurs inattendues : c'est weights @ V, dans cet ordre"
print("Exercice 3 validé : la moyenne pondérée des values est correcte.")
print(np.round(O, 2))

### Exercice 4 · Fabriquer le masque causal — niveau ●

`np.tril` (« triangular lower ») fabrique la matrice triangulaire inférieure.

In [ ]:
def masque_causal(n):
    """Masque causal (n, n) : 1 sur le triangle inférieur (passé + présent), 0 sur le futur."""
    masque = np.tril(np.ones((n, n), dtype=np.float32))
    return masque

In [ ]:
M = masque_causal(4)
assert M.shape == (4, 4), f"shape {M.shape} au lieu de (4, 4)"
assert np.allclose(M, np.tril(np.ones((4, 4)))), "le triangle inférieur (diagonale comprise) doit valoir 1"
assert M[0, 1] == 0 and M[3, 0] == 1, "1 = autorisé (passé), 0 = interdit (futur)"
print("Exercice 4 validé : le masque causal est correct.")
print(M.astype(int))

### Exercice 5 · Tout assembler : l'attention complète en NumPy — niveau ●●

Les quatre étapes plus le masque optionnel. Règle d'or du chapitre (section 4.3) : le masque
s'applique sur les **scores**, avec $-\infty$, **avant** le softmax. `np.where(mask == 0, -np.inf, scores)`
fait l'affaire : après l'exponentielle, ces positions pèsent exactement zéro.

In [ ]:
def attention(Q, K, V, mask=None):
    """Scaled dot-product attention complète, en NumPy.

    Retourne (output, weights) : output (n, d_v), weights (n, n).
    """
    scores = scores_attention(Q, K)
    if mask is not None:
        scores = np.where(mask == 0, -np.inf, scores)   # masque AVANT le softmax
    weights = softmax_stable(scores)
    output = melange_values(weights, V)
    return output, weights

In [ ]:
out, w = attention(Q, K, V)
assert np.allclose(out, O_attendu, atol=1e-2), "sans masque, on doit retrouver l'output de l'exercice 3"

out_c, w_c = attention(Q, K, V, mask=masque_causal(4))
OC_attendu = np.array([[10.0000, 0.0000, 0.0000, 0.0000],
              [3.7754, 12.4492, 0.0000, 0.0000],
              [3.5987, 7.1973, 8.4080, 0.0000],
              [2.1891, 4.3782, 8.4326, 11.2435]], dtype=np.float32)
assert np.allclose(w_c.sum(axis=1), 1.0, atol=1e-5), "chaque ligne doit encore sommer à 1"
assert np.allclose(w_c[np.triu_indices(4, k=1)], 0.0, atol=1e-6), "le futur doit peser exactement zéro"
assert np.allclose(out_c[0], V[0], atol=1e-4), "le token 0 ne voit que lui-même : son output = V[0]"
assert np.allclose(out_c, OC_attendu, atol=1e-2), "valeurs inattendues avec le masque causal"
print("Exercice 5 validé : attention complète, causale, et correctement renormalisée.")
print(np.round(w_c, 3))

### Exercice 6 · La fonction `scaled_dot_product_attention` — niveau ●●

La même logique qu'en NumPy, en gérant en plus le **batch** : Q, K, V sont en `(B, n, d_k)`.
`transpose(-2, -1)` n'échange que les deux dernières dimensions, la dimension de batch
traverse tout sans qu'on s'en occupe. C'est le code exact de la section 5 du chapitre ;
la validation compare ta version à la référence de PyTorch.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention en PyTorch, avec batch et masque optionnel.

    Q, K, V : (B, n, d_k)  ->  output (B, n, d_v), weights (B, n, n)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)   # (B, n, n)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))      # masque AVANT softmax
    weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

In [ ]:
torch.manual_seed(0)
Qb, Kb, Vb = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)

out, w = scaled_dot_product_attention(Qb, Kb, Vb)
assert out.shape == (2, 5, 8) and w.shape == (2, 5, 5), f"shapes : {out.shape}, {w.shape}"
ref = F.scaled_dot_product_attention(Qb, Kb, Vb)
assert torch.allclose(out, ref, atol=1e-5), "sans masque, ton résultat doit coller à la référence de PyTorch"

masque = torch.tril(torch.ones(5, 5))
out_c, w_c = scaled_dot_product_attention(Qb, Kb, Vb, mask=masque)
ref_c = F.scaled_dot_product_attention(Qb, Kb, Vb, is_causal=True)
assert torch.allclose(out_c, ref_c, atol=1e-5), "avec masque causal, idem"
assert torch.allclose(w_c.sum(dim=-1), torch.ones(2, 5), atol=1e-5)
print("Exercice 6 validé : ton attention PyTorch colle à F.scaled_dot_product_attention.")

### Exercice 7 · La classe `SelfAttention` — niveau ●●●

Il ne reste qu'à emballer ta fonction avec les trois projections apprises de la section 2.3 :
un même `x`, trois lectures. C'est le module qu'on branchera dans le Transformer au chapitre 10.

In [ ]:
class SelfAttention(nn.Module):
    """Self-attention : Q, K, V sont trois projections linéaires apprises du même x."""

    def __init__(self, d_model):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        Q = self.W_Q(x)                     # tout vient du même x : « self »-attention
        K = self.W_K(x)
        V = self.W_V(x)
        return scaled_dot_product_attention(Q, K, V, mask)

In [ ]:
torch.manual_seed(1)
attn = SelfAttention(d_model=16)
x = torch.randn(2, 5, 16)
masque = torch.tril(torch.ones(5, 5))

out, w = attn(x, mask=masque)
assert out.shape == (2, 5, 16) and w.shape == (2, 5, 5), f"shapes : {out.shape}, {w.shape}"
assert torch.allclose(w.sum(dim=-1), torch.ones(2, 5), atol=1e-5), "chaque ligne doit sommer à 1"
assert torch.allclose(w[:, 0, 1:], torch.zeros(2, 4), atol=1e-6), "causale : le token 0 ne voit que lui-même"

out.sum().backward()
assert attn.W_Q.weight.grad is not None and torch.isfinite(attn.W_Q.weight.grad).all(), \
    "les gradients doivent circuler jusqu'aux projections"
print("Exercice 7 validé : ta SelfAttention est complète, causale et entraînable.")
print("Pacte tenu. Tu viens d'écrire le cœur des LLMs.")